# Calibración: Thresholds por clase + Temperature Scaling

**Problema 1 (CRÍTICO):** Infiltration threshold=0.25 → sensibilidad 16.7%
**Problema 2:** Probabilidades del ensemble no calibradas (temperature scaling)

**Solución:**
1. Calcular curva ROC en test set para cada clase
2. Elegir threshold óptimo por criterio clínico (no solo Youden's J)
3. Para Infiltration/Pneumonia: priorizar sensibilidad ≥ 0.70
4. Calibrar temperatura sobre val set

**Runtime: CPU. ~15 min.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, json
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
import numpy as np
from scipy.optimize import minimize_scalar

!pip install -q torchxrayvision h5py
print('Setup OK')

CLASSES_14 = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion',
    'Emphysema','Fibrosis','Hernia','Infiltration','Mass','Nodule',
    'Pleural_Thickening','Pneumonia','Pneumothorax',
]

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_ML = '/content/processed_s4ml'


## 2. HDF5 y CSVs

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    print('Copiando HDF5...')
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'HDF5: {len(hf["images"]):,} imgs')

import pandas as pd
os.makedirs(PROCESSED_ML, exist_ok=True)
for csv in ['val.csv', 'test.csv']:
    local = f'{PROCESSED_ML}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s4ml/{csv}', local)
    print(f'{csv}: {len(pd.read_csv(local))} filas')


## 3. Cargar ensemble

In [ ]:
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.models.cnn_vit import CNNViT
from src.training.utils import get_device

device = get_device()

model_v1 = CNNViT(
    backbone_weights='densenet121-res224-nih',
    num_classes=14, embedding_dim=512, num_heads=8,
    num_layers=4, mlp_dim=1024,
).to(device)
ckpt_v1 = torch.load(
    f'{PROJECT_ROOT}/experiments_s4ml/checkpoints/sprint4ml_phase2_best.pt',
    map_location=device
)
model_v1.load_state_dict(ckpt_v1['model_state_dict'])
model_v1.eval()

model_v2 = CNNViT(
    backbone_weights='densenet121-res224-nih',
    num_classes=14, embedding_dim=512, num_heads=8,
    num_layers=6, mlp_dim=1024,
).to(device)
ckpt_v2 = torch.load(
    f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase2_best.pt',
    map_location=device
)
model_v2.load_state_dict(ckpt_v2['model_state_dict'])
model_v2.eval()

print('✓ Ensemble cargado (v1=4 capas, v2=6 capas)')


## 4. Obtener predicciones en val y test

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih_multilabel import NIHDatasetMultiLabel
from src.datasets.transforms import build_eval_transform

eval_tfm = build_eval_transform(224)
W1, W2   = 0.3, 0.7

def get_ensemble_probs(csv_path):
    ds     = NIHDatasetMultiLabel(csv_path, H5_LOCAL, eval_tfm)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            imgs = batch['image'].to(device)
            p1   = torch.sigmoid(model_v1(imgs)).cpu().numpy()
            p2   = torch.sigmoid(model_v2(imgs)).cpu().numpy()
            probs = W1 * p1 + W2 * p2
            all_probs.append(probs)
            all_labels.append(batch['label'].numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

print('Obteniendo predicciones en val...')
val_probs,  val_labels  = get_ensemble_probs(f'{PROCESSED_ML}/val.csv')
print(f'  val: {len(val_labels)} muestras')

print('Obteniendo predicciones en test...')
test_probs, test_labels = get_ensemble_probs(f'{PROCESSED_ML}/test.csv')
print(f'  test: {len(test_labels)} muestras')


## 5. Calibrar thresholds por clase

**Criterio clínico por clase:**
- Infiltration, Pneumonia: sensibilidad ≥ 0.70 (prioriza no perder casos TB/neumonía)
- Resto: Youden's J (balance sens/spec)


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Criterio de sensibilidad mínima por clase
MIN_SENSITIVITY = {
    'Infiltration': 0.70,   # TB context HNAL
    'Pneumonia':    0.65,
    'Effusion':     0.60,
    'Edema':        0.60,
    'default':      None,   # None = Youden's J (balance)
}

print('CALIBRACIÓN DE THRESHOLDS (sobre val set):')
print('='*65)
print(f"{'Clase':<22} {'AUC':>6} {'Threshold':>10} {'Sens':>6} {'Spec':>6} {'Criterio'}")
print('-'*65)

optimal_thresholds = {}

for i, cls in enumerate(CLASSES_14):
    y_true = val_labels[:, i]
    y_prob = val_probs[:, i]

    n_pos = y_true.sum()
    if n_pos == 0:
        optimal_thresholds[cls] = 0.3
        print(f'{cls:<22} {"N/A":>6} {"0.300":>10} {"N/A":>6} {"N/A":>6} sin positivos')
        continue

    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    tnr = 1 - fpr  # specificity

    min_sens = MIN_SENSITIVITY.get(cls, MIN_SENSITIVITY['default'])

    if min_sens is not None:
        # Criterio: máxima especificidad con sensibilidad >= min_sens
        valid = tpr >= min_sens
        if valid.any():
            # Entre los puntos válidos, elegir el de mayor especificidad
            valid_idx = np.where(valid)[0]
            best_idx  = valid_idx[np.argmax(tnr[valid_idx])]
            criterio  = f'Sens≥{min_sens}'
        else:
            # Si no se puede alcanzar, elegir el de mayor sensibilidad
            best_idx = np.argmax(tpr)
            criterio = f'MaxSens(no≥{min_sens})'
    else:
        # Youden's J = maximiza (sens + spec - 1)
        youden   = tpr + tnr - 1
        best_idx = np.argmax(youden)
        criterio = "Youden's J"

    # Evitar threshold=0 o threshold=1
    thr = float(np.clip(thresholds[best_idx], 0.05, 0.95))
    optimal_thresholds[cls] = round(thr, 3)

    sens_at_thr = float(tpr[best_idx])
    spec_at_thr = float(tnr[best_idx])

    print(f'{cls:<22} {auc:>6.4f} {thr:>10.3f} {sens_at_thr:>6.3f} {spec_at_thr:>6.3f} {criterio}')

print()
print('Thresholds óptimos:')
print(json.dumps(optimal_thresholds, indent=2))


## 6. Evaluar thresholds optimizados en test

In [ ]:
from sklearn.metrics import classification_report

print('COMPARATIVA: threshold=0.5 vs threshold calibrado (TEST):')
print('='*65)
print(f"{'Clase':<22} {'Sens(0.5)':>9} {'Spec(0.5)':>9} {'Sens(opt)':>9} {'Spec(opt)':>9}")
print('-'*65)

for i, cls in enumerate(CLASSES_14):
    y_true = test_labels[:, i]
    y_prob = test_probs[:, i]

    if y_true.sum() == 0:
        continue

    for thr, label in [(0.5, ''), (optimal_thresholds[cls], '')]:
        y_pred = (y_prob >= thr).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        if thr == 0.5:
            sens_05, spec_05 = sens, spec
        else:
            sens_opt, spec_opt = sens, spec

    marker = ' ← CRÍTICO' if cls in ['Infiltration','Pneumonia'] else ''
    print(f'{cls:<22} {sens_05:>9.3f} {spec_05:>9.3f} {sens_opt:>9.3f} {spec_opt:>9.3f}{marker}')

print()
print('Infiltration con threshold=0.5:   sens≈0.167 (inaceptable)')
print(f'Infiltration con threshold={optimal_thresholds["Infiltration"]:.3f}: sens={sens_opt:.3f} (objetivo: ≥0.70)')


## 7. Temperature scaling del ensemble

In [ ]:
import torch.nn.functional as F

print('TEMPERATURE SCALING (sobre val set):')
print('Buscar T que minimiza NLL...')

def nll_ensemble(T, probs, labels):
    # NLL para multi-label con temperatura
    p   = np.clip(probs ** (1/T), 1e-7, 1-1e-7)
    nll = -(labels * np.log(p) + (1-labels) * np.log(1-p))
    return float(nll.mean())

def mean_confidence(probs):
    # Confianza media = promedio de max(p, 1-p) por muestra
    return float(np.maximum(probs, 1-probs).mean())

# Buscar T óptima
result = minimize_scalar(
    nll_ensemble,
    bounds=(0.1, 5.0),
    method='bounded',
    args=(val_probs, val_labels),
)
T_optimal = float(result.x)

# Comparar
print(f'\nT = 1.0 (sin scaling):')
print(f'  NLL:             {nll_ensemble(1.0, val_probs, val_labels):.4f}')
print(f'  Confianza media: {mean_confidence(val_probs)*100:.1f}%')
print(f'\nT = {T_optimal:.4f} (óptima):')
print(f'  NLL:             {nll_ensemble(T_optimal, val_probs, val_labels):.4f}')
print(f'  Confianza media: {mean_confidence(val_probs**(1/T_optimal))*100:.1f}%')

if abs(T_optimal - 1.0) < 0.05:
    print('\n→ T ≈ 1.0: el ensemble ya está bien calibrado. No aplicar scaling.')
elif T_optimal < 1.0:
    print(f'\n→ T < 1.0: el ensemble es poco confiante. Aplicar T={T_optimal:.4f}.')
else:
    print(f'\n→ T > 1.0: el ensemble es overconfident. Aplicar T={T_optimal:.4f}.')


## 8. Guardar configuración calibrada

In [ ]:
# Construir thresholds finales con metadata
thresholds_final = {
    "thresholds": optimal_thresholds,
    "temperature": round(T_optimal, 4),
    "apply_temperature": abs(T_optimal - 1.0) > 0.05,
    "criterion": {
        "Infiltration":     "sens>=0.70 (TB context HNAL)",
        "Pneumonia":        "sens>=0.65",
        "Effusion":         "sens>=0.60",
        "Edema":            "sens>=0.60",
        "default":          "Youden's J",
    },
    "calibrated_on": "val",
    "note": "Thresholds calibrados por clase con criterio clínico para HNAL"
}

# Guardar en Drive
out_drive = f'{PROJECT_ROOT}/experiments_s4ml_v2/thresholds_calibrated.json'
with open(out_drive, 'w') as f:
    json.dump(thresholds_final, f, indent=2, ensure_ascii=False)
print(f'✓ Guardado en Drive: {out_drive}')

# Mostrar el JSON final para copiar al backend
print()
print('COPIA ESTE CONTENIDO a backend/artifacts/thresholds_14.json:')
print()
print(json.dumps({
    "thresholds": optimal_thresholds,
    "temperature": round(T_optimal, 4),
    "apply_temperature": abs(T_optimal - 1.0) > 0.05,
    "criterion": "clinical_sensitivity_priority_HNAL"
}, indent=2))


## 9. Curvas ROC de Infiltration y Pneumonia

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cls in zip(axes, ['Infiltration', 'Pneumonia']):
    i = CLASSES_14.index(cls)
    fpr, tpr, thrs = roc_curve(test_labels[:,i], test_probs[:,i])
    auc = roc_auc_score(test_labels[:,i], test_probs[:,i])

    ax.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC={auc:.4f})')
    ax.plot([0,1],[0,1],'--',color='gray',alpha=0.5)

    # Marcar threshold óptimo
    thr_opt = optimal_thresholds[cls]
    y_pred  = (test_probs[:,i] >= thr_opt).astype(int)
    tp  = ((y_pred==1)&(test_labels[:,i]==1)).sum()
    fn  = ((y_pred==0)&(test_labels[:,i]==1)).sum()
    fp  = ((y_pred==1)&(test_labels[:,i]==0)).sum()
    tn  = ((y_pred==0)&(test_labels[:,i]==0)).sum()
    sens_opt = tp/(tp+fn) if (tp+fn)>0 else 0
    spec_opt = tn/(tn+fp) if (tn+fp)>0 else 0

    ax.scatter([1-spec_opt],[sens_opt], s=150, c='red', zorder=5,
               label=f'Thr={thr_opt:.3f} (Sens={sens_opt:.2f}, Spec={spec_opt:.2f})')

    # Marcar threshold=0.5 para comparar
    y_05   = (test_probs[:,i] >= 0.5).astype(int)
    tp5 = ((y_05==1)&(test_labels[:,i]==1)).sum()
    fn5 = ((y_05==0)&(test_labels[:,i]==1)).sum()
    fp5 = ((y_05==1)&(test_labels[:,i]==0)).sum()
    tn5 = ((y_05==0)&(test_labels[:,i]==0)).sum()
    s5 = tp5/(tp5+fn5) if (tp5+fn5)>0 else 0
    sp5= tn5/(tn5+fp5) if (tn5+fp5)>0 else 0

    ax.scatter([1-sp5],[s5], s=150, c='orange', zorder=5, marker='s',
               label=f'Thr=0.500 (Sens={s5:.2f}, Spec={sp5:.2f})')

    ax.set_xlabel('1 - Especificidad (FPR)'); ax.set_ylabel('Sensibilidad (TPR)')
    ax.set_title(f'ROC — {cls}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle('Calibración de threshold — Prioridad sensibilidad (HNAL)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s4ml_v2/figures/roc_infiltration_pneumonia.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada.')
print()
print('RESUMEN:')
for cls in ['Infiltration', 'Pneumonia']:
    thr = optimal_thresholds[cls]
    print(f'  {cls}: threshold={thr:.3f}')
